<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z340_NormalizacionSeries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon con normalización de series

El problema: AutoGluon ve todos los productos juntos. Si un producto vende 1 tn y otro 1000 tn, el modelo aprende más del producto grande y el chico queda tapado.

La idea: normalizar cada serie antes de dársela a AutoGluon → todos los productos quedan en la misma escala → AutoGluon aprende patrones de **forma** (sube, baja, estable) independientemente del nivel. Después desnormalizamos la predicción.

## Las 3 normalizaciones

**1. Norm max** — `serie / max(serie)` → escala [0,1], el pico histórico siempre vale 1.0. Robusto a ceros.

**2. Norm L2** — `serie / ||serie||` → vector unitario. Captura la distribución relativa de ventas entre períodos sin importar el nivel absoluto.

**3. Norm index** — `serie / media_primeros_3m` → todas las series arrancan en ~1.0. Captura el cambio relativo al nivel inicial del producto.

## Pipeline
```
serie original  →  normalizar  →  AutoGluon  →  pred normalizada  →  × escala  →  pred real
```

Backtesting: AutoGluon entrena hasta 201910, predice 201912 internamente.
Submit: reentrenar con toda la historia hasta 201912, predecir 202002.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q
!uv pip install -q kaggle autogluon.timeseries

In [ ]:
import os, shutil
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'competencia':      'labo-iii-2026-rosario',
    'periodo_corte':    201910,   # hasta acá para backtesting
    'periodo_target':   201912,   # valor real conocido para evaluar
    'horizonte':        2,        # predecir t+2
    'ag_tiempo':        60 * 5,   # segundos que tiene AutoGluon por normalización
    'drive_path':       '/content/buckets/b1/exp/NormalizacionSeries',
}

os.makedirs(PARAM['drive_path'], exist_ok=True)

dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_ventas    = dataset.group_by('product_id','periodo').agg(pl.col('tn').sum()).sort(['product_id','periodo'])
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')
tb_ventas    = tb_ventas.join(tb_apredecir, on='product_id', how='inner').sort(['product_id','periodo'])
productos    = tb_apredecir['product_id'].to_list()
print(f'{len(productos)} productos')

# Funciones de normalización

In [ ]:
def norm_max(serie: np.ndarray):
    """Divide por el máximo histórico. Escala [0,1]. Robusto a ceros."""
    m = serie.max()
    return (serie / m, float(m)) if m > 0 else (serie.copy(), 1.0)


def norm_l2(serie: np.ndarray):
    """Divide por la norma L2. Vector unitario. Robusto a ceros."""
    norma = float(np.sqrt((serie ** 2).sum()))
    return (serie / norma, norma) if norma > 0 else (serie.copy(), 1.0)


def norm_index(serie: np.ndarray, n_base: int = 3):
    """Divide por la media de los primeros n_base meses con venta > 0.
    Todas las series arrancan en ~1.0."""
    positivos = serie[serie > 0]
    base = float(positivos[:n_base].mean()) if len(positivos) > 0 else 1.0
    return (serie / base, base) if base > 0 else (serie.copy(), 1.0)


NORMALIZACIONES = {
    'max':   norm_max,
    'l2':    norm_l2,
    'index': norm_index,
}
print('OK')

# Construir TimeSeriesDataFrame normalizado

AutoGluon necesita un dataframe largo con columnas `item_id`, `timestamp`, `target`.
Guardamos también la escala por producto para desnormalizar después.

In [ ]:
def build_tsdf(tb, productos, norm_fn, periodo_max=None):
    """
    Construye el TimeSeriesDataFrame normalizado y devuelve también
    el dict {product_id: escala} para desnormalizar.
    """
    rows  = []
    escala_dict = {}

    for pid in productos:
        df = tb.filter(pl.col('product_id') == pid).sort('periodo')
        if periodo_max is not None:
            df = df.filter(pl.col('periodo') <= periodo_max)

        serie    = df['tn'].to_numpy().astype(float)
        periodos = df['periodo'].to_list()

        if len(serie) == 0:
            continue

        serie_norm, escala = norm_fn(serie)
        escala_dict[pid]   = escala

        for p, v in zip(periodos, serie_norm):
            # periodo YYYYMM → timestamp: primer día del mes
            year, month = int(str(p)[:4]), int(str(p)[4:])
            rows.append({
                'item_id':   str(pid),
                'timestamp': f'{year}-{month:02d}-01',
                'target':    float(v),
            })

    df_pd = pl.DataFrame(rows).to_pandas()
    df_pd['timestamp'] = pl.from_pandas(df_pd[['timestamp']]).with_columns(
        pl.col('timestamp').str.to_datetime()
    )['timestamp'].to_pandas()

    tsdf = TimeSeriesDataFrame.from_data_frame(
        df_pd,
        id_column='item_id',
        timestamp_column='timestamp',
    )
    return tsdf, escala_dict


print('OK')

# Backtesting con AutoGluon — las 3 normalizaciones

In [ ]:
tb_real = (
    tb_ventas.filter(pl.col('periodo') == PARAM['periodo_target'])
    .select(['product_id','tn']).rename({'tn':'tn_real'})
)
reales = {row['product_id']: row['tn_real'] for row in tb_real.to_dicts()}

rmse_bt      = {}
preds_bt     = {}   # guardamos para graficar

for nombre, fn in NORMALIZACIONES.items():
    print(f'\n══ Backtesting: norm={nombre} ══')

    tsdf, escala_dict = build_tsdf(
        tb_ventas, productos, fn,
        periodo_max=PARAM['periodo_corte']
    )

    ruta_modelo = f"/tmp/ag_bt_{nombre}"
    predictor = TimeSeriesPredictor(
        path=ruta_modelo,
        prediction_length=PARAM['horizonte'],
        target='target',
        eval_metric='RMSE',
        verbosity=1,
    )
    predictor.fit(
        tsdf,
        time_limit=PARAM['ag_tiempo'],
        presets='medium_quality',
    )

    # predecir horizonte=2 pasos
    preds_norm = predictor.predict(tsdf)

    # desnormalizar y calcular RMSE
    errores = []
    pred_list = []
    for pid in productos:
        item = str(pid)
        if item not in preds_norm.index.get_level_values('item_id'):
            continue
        # t+2 es la segunda fila del horizonte
        pred_norm = float(preds_norm.loc[item]['mean'].iloc[-1])
        pred_norm = max(pred_norm, 0.0)
        pred_real = pred_norm * escala_dict.get(pid, 1.0)
        real_val  = reales.get(pid, np.nan)
        if not np.isnan(real_val):
            errores.append((pred_real - real_val) ** 2)
        pred_list.append({'product_id': pid, 'tn': pred_real})

    rmse = float(np.sqrt(np.mean(errores)))
    rmse_bt[nombre] = rmse
    preds_bt[nombre] = pred_list
    print(f'  RMSE backtesting 201912 (norm={nombre}): {rmse:.4f}')

print('\n── Resumen backtesting ──')
for n, r in sorted(rmse_bt.items(), key=lambda x: x[1]):
    print(f'  {n:8s}: {r:.4f}')

# Submit — reentrenar con toda la historia hasta 201912 y predecir 202002

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

for nombre, fn in NORMALIZACIONES.items():
    print(f'\n══ Submit: norm={nombre} ══')

    tsdf_full, escala_dict = build_tsdf(
        tb_ventas, productos, fn,
        periodo_max=201912   # toda la historia disponible
    )

    ruta_modelo = f"/tmp/ag_submit_{nombre}"
    predictor = TimeSeriesPredictor(
        path=ruta_modelo,
        prediction_length=PARAM['horizonte'],
        target='target',
        eval_metric='RMSE',
        verbosity=1,
    )
    predictor.fit(
        tsdf_full,
        time_limit=PARAM['ag_tiempo'],
        presets='medium_quality',
    )

    preds_norm = predictor.predict(tsdf_full)

    preds_final = []
    for pid in productos:
        item = str(pid)
        if item not in preds_norm.index.get_level_values('item_id'):
            preds_final.append({'product_id': pid, 'tn': 0.0})
            continue
        pred_norm = float(preds_norm.loc[item]['mean'].iloc[-1])
        pred_norm = max(pred_norm, 0.0)
        pred_real = pred_norm * escala_dict.get(pid, 1.0)
        preds_final.append({'product_id': pid, 'tn': pred_real})

    tb_final = pl.DataFrame(preds_final)
    archivo  = f'ag_norm_{nombre}.csv'
    mensaje  = f'AutoGluon norm={nombre} RMSE_bt={rmse_bt.get(nombre, 0):.4f}'

    tb_final.write_csv(archivo)
    shutil.copy(archivo, f"{PARAM['drive_path']}/{archivo}")
    kaggle_submit(PARAM['competencia'], archivo, mensaje)
    print(f'  submitted + guardado en Drive: {archivo}')